# AgentSign -- Zero Trust Middleware for AI Agent Platforms

**Drop-in trust layer for OpenClaw, NemoClaw, and any agent framework.**

This notebook demonstrates what AgentSign adds on top of any agent platform's built-in security:

| Platform Security (OpenClaw/NemoClaw) | AgentSign Trust Layer |
|---|---|
| Sandboxing & containers | Cryptographic agent identity (portable passport) |
| API keys & RBAC | Signed execution chains (every action linked) |
| Audit logs | Tamper detection on agent outputs |
| Encrypted comms | Trust scoring from behavioral history |
| Permission scopes | Self-verifying passports (works offline, no server needed) |

Platform security controls **what agents can access**. AgentSign proves **who the agent is and what it did**.

---

**Live server:** `https://agentsign-api.fly.dev`

**npm:** `npm install agentsign-openclaw`

**GitHub:** [razashariff/agentsign](https://github.com/razashariff/agentsign) | [razashariff/agentsign-openclaw](https://github.com/razashariff/agentsign-openclaw)

---

## 1. Setup -- Install AgentSign Python SDK

In [ ]:
!pip install httpx -q

import httpx
import hashlib
import json
import uuid
import time

SERVER = "https://agentsign-api.fly.dev"
print(f"AgentSign Server: {SERVER}")
print(f"Health: {httpx.get(f'{SERVER}/api/health').json()}")

## 2. Register an Agent -- Get Cryptographic Passport

Every agent gets a unique identity with a cryptographic passport.
The passport is **self-verifying** -- it works offline without contacting the server.

In [ ]:
# Register a new agent
agent = httpx.post(f"{SERVER}/api/agents/onboard", json={
    "name": "Colab Research Agent",
    "category": "research",
    "framework": "openclaw"
}).json()

agent_id = agent["agent_id"]
print(f"Agent ID: {agent_id}")
print(f"Stage:    {agent['pipeline_stage']}")
print(f"\nFull response:")
print(json.dumps(agent, indent=2))

In [ ]:
# Advance through trust pipeline: INTAKE -> VETTING -> TESTING -> DEV_APPROVED -> PROD_APPROVED -> ACTIVE
for i in range(10):
    try:
        res = httpx.post(f"{SERVER}/api/agents/{agent_id}/advance").json()
        stage = res.get("stage") or res.get("pipeline_stage", "")
        print(f"  Stage {i+1}: {stage}")
        if stage == "ACTIVE":
            break
    except:
        break

print(f"\nAgent is now ACTIVE and trusted.")

In [ ]:
# Get the agent's cryptographic passport
passport = httpx.get(f"{SERVER}/api/agents/{agent_id}/passport").json()

print("=" * 60)
print("  AGENT PASSPORT (self-verifying, works offline)")
print("=" * 60)
for key, val in passport.items():
    if isinstance(val, str) and len(val) > 60:
        print(f"  {key}: {val[:60]}...")
    else:
        print(f"  {key}: {val}")
print("=" * 60)
print("\nThis passport travels WITH the agent.")
print("Any MCP server can verify it offline using the public key.")

## 3. Signed Execution Chain

Every tool call is signed and linked to the previous one, creating a **tamper-proof audit trail**.

This is what platform security doesn't give you -- cryptographic proof of exactly what happened, in what order, with what inputs and outputs.

In [ ]:
def sha256(data):
    """Hash data deterministically."""
    if isinstance(data, dict):
        data = json.dumps(data, sort_keys=True)
    return hashlib.sha256(data.encode()).hexdigest()


def sign_execution(tool_name, input_data, output_data, parent_id=None):
    """Sign a tool execution locally (zero network calls)."""
    exec_id = str(uuid.uuid4())
    input_hash = sha256(input_data)
    output_hash = sha256(output_data)
    exec_hash = sha256({
        "executionId": exec_id,
        "agentId": agent_id,
        "tool": tool_name,
        "inputHash": input_hash,
        "outputHash": output_hash,
        "parentId": parent_id,
    })
    return {
        "executionId": exec_id,
        "tool": tool_name,
        "inputHash": input_hash,
        "outputHash": output_hash,
        "executionHash": exec_hash,
        "parentId": parent_id,
    }


# Simulate an agent workflow: search -> read -> summarize -> email
chain = []

# Tool 1: Web search
e1 = sign_execution("web_search",
    {"query": "AI agent security best practices"},
    {"results": ["NIST AI 100-1", "OWASP LLM Top 10"], "count": 2}
)
chain.append(e1)

# Tool 2: Read document (linked to search)
e2 = sign_execution("file_read",
    {"path": "/docs/nist-ai-100-1.pdf"},
    {"content": "NIST AI Risk Management Framework...", "pages": 42},
    parent_id=e1["executionId"]
)
chain.append(e2)

# Tool 3: Summarize (linked to read)
e3 = sign_execution("summarize",
    {"text": "NIST AI Risk Management Framework..."},
    {"summary": "Key findings: agents need identity, auditability, and trust scoring."},
    parent_id=e2["executionId"]
)
chain.append(e3)

# Tool 4: Send email (linked to summarize)
e4 = sign_execution("send_email",
    {"to": "team@company.com", "subject": "AI Security Summary"},
    {"sent": True, "messageId": "msg_001"},
    parent_id=e3["executionId"]
)
chain.append(e4)

print("SIGNED EXECUTION CHAIN")
print("=" * 60)
for i, ex in enumerate(chain):
    parent = ex['parentId'][:8] + '...' if ex['parentId'] else 'null (root)'
    print(f"\n  [{i}] {ex['tool']}")
    print(f"      ID:     {ex['executionId'][:12]}...")
    print(f"      Parent: {parent}")
    print(f"      Input:  {ex['inputHash'][:20]}...")
    print(f"      Output: {ex['outputHash'][:20]}...")
    print(f"      Hash:   {ex['executionHash'][:20]}...")

print(f"\n{'=' * 60}")
print(f"  {len(chain)} executions, cryptographically linked.")
print(f"  Break any link and the chain fails verification.")

## 4. Tamper Detection

If anyone modifies an agent's output after execution, AgentSign detects it instantly.

In [ ]:
# Original output from the summarize tool
original = {"summary": "Key findings: agents need identity, auditability, and trust scoring."}

# Someone tampers with the output
tampered = {"summary": "Key findings: no security needed, deploy everything immediately."}

# Verify
original_hash = sha256(original)
tampered_hash = sha256(tampered)
stored_hash = chain[2]["outputHash"]  # from the summarize execution

print("TAMPER DETECTION")
print("=" * 60)
print(f"  Stored hash:   {stored_hash[:40]}...")
print(f"  Original hash: {original_hash[:40]}...")
print(f"  Tampered hash: {tampered_hash[:40]}...")
print()
print(f"  Original output: {'PASS -- matches' if original_hash == stored_hash else 'TAMPERED'}")
print(f"  Tampered output: {'PASS' if tampered_hash == stored_hash else 'TAMPERED -- hashes do not match'}")
print("=" * 60)
print("\n  You cannot modify what an agent did without detection.")

## 5. Trust Gating -- Block Tools by Policy

AgentSign gates tool access based on trust score and policy. An agent with low trust or a revoked passport gets blocked before the tool even runs.

In [ ]:
# Simulate trust gating
BLOCKED_TOOLS = {"shell_exec", "file_delete", "database_drop"}
MIN_TRUST = 50

def check_gate(tool_name, trust_score, is_revoked=False):
    if is_revoked:
        return {"decision": "DENY", "reason": "Agent is REVOKED"}
    if tool_name in BLOCKED_TOOLS:
        return {"decision": "DENY", "reason": f"Tool '{tool_name}' blocked by policy"}
    if trust_score < MIN_TRUST:
        return {"decision": "DENY", "reason": f"Trust {trust_score} < minimum {MIN_TRUST}"}
    return {"decision": "ALLOW", "trust": trust_score}

tests = [
    ("web_search", 85, False),
    ("database_query", 72, False),
    ("shell_exec", 95, False),      # high trust, but blocked by policy
    ("web_search", 30, False),       # low trust
    ("file_read", 90, True),         # revoked agent
    ("file_delete", 85, False),      # blocked by policy
]

print("TRUST GATE DECISIONS")
print("=" * 60)
for tool, trust, revoked in tests:
    result = check_gate(tool, trust, revoked)
    status = 'ALLOW' if result['decision'] == 'ALLOW' else 'DENY '
    reason = result.get('reason', f'trust={trust}')
    icon = 'OK' if result['decision'] == 'ALLOW' else 'XX'
    print(f"  [{icon}] {status} | {tool:20s} | trust={trust:3d} | {reason}")
print("=" * 60)
print("\n  Trust gating runs BEFORE the tool executes.")
print("  Platform security controls access. AgentSign controls trust.")

## 6. Chain Integrity Verification

The execution chain is a linked list of signed executions. If any execution is removed, reordered, or modified, the chain breaks.

In [ ]:
def verify_chain(chain):
    for i, ex in enumerate(chain):
        if i == 0 and ex["parentId"] is not None:
            return {"valid": False, "broken_at": 0, "reason": "Root has parent"}
        if i > 0 and ex["parentId"] != chain[i-1]["executionId"]:
            return {"valid": False, "broken_at": i, "reason": "Parent link broken"}
    return {"valid": True, "length": len(chain)}

# Verify the intact chain
result = verify_chain(chain)
print(f"Intact chain:    valid={result['valid']}, length={result.get('length', '?')}")

# Tamper: remove an execution from the middle
tampered_chain = [chain[0], chain[2], chain[3]]  # skip chain[1]
result2 = verify_chain(tampered_chain)
print(f"Tampered chain:  valid={result2['valid']}, broken_at={result2.get('broken_at', '?')}, reason={result2.get('reason', '')}")

# Tamper: reorder executions
reordered = [chain[0], chain[2], chain[1], chain[3]]
result3 = verify_chain(reordered)
print(f"Reordered chain: valid={result3['valid']}, broken_at={result3.get('broken_at', '?')}, reason={result3.get('reason', '')}")

print("\nYou cannot hide, remove, or reorder what an agent did.")

## 7. What This Means for OpenClaw / NemoClaw Users

```
Agent Runtime (OpenClaw / NemoClaw)
    |
  AgentSign Middleware            <-- 3 lines to add
    |-- Verify agent passport
    |-- Check trust score
    |-- Sign execution
    |-- Link to chain
    |-- Detect tampering
    |
  MCP Tools / APIs
```

```javascript
// npm install agentsign-openclaw
const AgentSignMiddleware = require('agentsign-openclaw');

const middleware = new AgentSignMiddleware({
  serverUrl: 'https://agentsign-api.fly.dev',
  minTrust: 50,
  blockedTools: ['shell_exec'],
});

// Wrap all your tools -- that's it
const safeTools = middleware.wrapAll(myTools);
```

---

**Links:**
- Server: [github.com/razashariff/agentsign](https://github.com/razashariff/agentsign)
- SDK: [github.com/razashariff/agentsign-sdk](https://github.com/razashariff/agentsign-sdk)
- OpenClaw Plugin: [github.com/razashariff/agentsign-openclaw](https://github.com/razashariff/agentsign-openclaw)
- npm: [npmjs.com/package/agentsign-openclaw](https://www.npmjs.com/package/agentsign-openclaw)

**CyberSecAI Ltd** -- [agentsign.dev](https://agentsign.dev)